# Archived research notebook

Original filename: `preprocessing first注释.ipynb`  
Purpose: Annotate CHB-MIT recordings from seizure summaries.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
#没用代码，检验是否能成功读取注释


import os
from collections import defaultdict

# 解析摘要文件获取癫痫发作信息
def parse_seizure_summary(summary_path):
    seizure_info = defaultdict(list)
    current_file = None
    with open(summary_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # 提取EDF文件名
            if line.startswith('File Name:'):
                file_name = line.split(': ')[1].split('.edf')[0] + '.edf'
                current_file = file_name
            # 提取发作起始和结束时间
            elif line.startswith('Seizure Start Time:') and current_file:
                start_sec = float(line.split(': ')[1].split(' seconds')[0])
            elif line.startswith('Seizure End Time:') and current_file:
                end_sec = float(line.split(': ')[1].split(' seconds')[0])
                seizure_info[current_file].append((start_sec, end_sec))
    return seizure_info

# 获取所有EDF文件路径并关联发作信息
def get_edf_seizure_pairs(folder_path, summary_path):
    # 解析摘要文件
    seizure_info = parse_seizure_summary(summary_path)
    # 遍历文件夹获取EDF文件路径
    edf_files = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.edf'):
                edf_path = os.path.join(root, file)
                edf_name = os.path.basename(edf_path)
                # 关联发作信息
                seizures = seizure_info.get(edf_name, [])
                edf_files.append({
                    'path': edf_path,
                    'seizures': seizures,
                    'file_name': edf_name
                })
    return edf_files

# 示例用法
if __name__ == "__main__":
    folder_path = r"data/chb-mit/chb01"  # EDF文件所在文件夹
    summary_path = r"data/chb-mit/chb01/chb01-summary.txt"  # 摘要文件路径
    # 获取所有EDF文件及其发作信息
    edf_seizure_pairs = get_edf_seizure_pairs(folder_path, summary_path)
    # 打印结果示例
    for item in edf_seizure_pairs:
        print(f"文件: {item['file_name']}")
        if item['seizures']:
            print(f" 发作次数: {len(item['seizures'])}")
            for idx, (start, end) in enumerate(item['seizures'], 1):
                print(f" 发作 {idx}: 开始时间 {start}s, 结束时间 {end}s")
        else:
            print(" 无发作信息")
        print("-" * 50)

In [ ]:
import os
import mne
from collections import defaultdict
from datetime import datetime  # 用于判断类型

# 解析摘要文件获取癫痫发作信息
def parse_seizure_summary(summary_path):
    seizure_info = defaultdict(list)
    current_file = None

    with open(summary_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith('File Name:'):
                file_name = line.split(': ')[1].split('.edf')[0] + '.edf'
                current_file = file_name
            elif line.startswith('Seizure Start Time:') and current_file:
                start_sec = float(line.split(': ')[1].split(' seconds')[0])
            elif line.startswith('Seizure End Time:') and current_file:
                end_sec = float(line.split(': ')[1].split(' seconds')[0])
                seizure_info[current_file].append((start_sec, end_sec))

    return seizure_info

# 为 EDF 文件添加发作注释并保存为 FIF 文件
def annotate_edf_files(folder_path, summary_path):
    seizure_info = parse_seizure_summary(summary_path)

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.edf'):
                edf_path = os.path.join(root, file)
                edf_name = file
                seizures = seizure_info.get(edf_name, [])

                if not seizures:
                    print(f"跳过 {edf_name}：无发作信息")
                    continue

                try:
                    # 读取 EDF 文件（preload=True 确保注释正确关联时间）
                    raw = mne.io.read_raw_edf(edf_path, preload=True)
                    
                    # -------------------- 处理 datetime 类型的 meas_date --------------------
                    meas_date = raw.info.get('meas_date')
                    if isinstance(meas_date, datetime):
                        # 将 datetime 对象转换为 Unix 时间戳（整数秒）
                        meas_date = int(meas_date.timestamp())
                        print(f"检测到 datetime 时间戳，转换为 {meas_date}")
                    elif isinstance(meas_date, (float, int)):
                        meas_date = int(meas_date)  # 确保为整数
                    else:
                        meas_date = 0  # 其他情况设为0

                    # 校验时间戳是否在有效范围内
                    if not (-2147483648 <= meas_date <= 2147483647):
                        print(f"重置无效时间戳 {meas_date} 为 0")
                        meas_date = 0
                    raw.set_meas_date(meas_date)
                    # -------------------- 其他逻辑保持不变 --------------------

                    # 创建注释对象（使用相对时间，基于文件开头的秒数）
                    onsets = [start for start, end in seizures]
                    durations = [end - start for start, end in seizures]
                    descriptions = ['seizure'] * len(seizures)
                    annotations = mne.Annotations(
                        onset=onsets,
                        duration=durations,
                        description=descriptions,
                        orig_time=meas_date  # 使用转换后的整数时间戳
                    )

                    # 添加注释
                    raw.set_annotations(annotations)
                    print(f"成功为 {edf_name} 添加 {len(seizures)} 条发作注释")

                    # 保存为符合 MNE 规范的文件名
                    fif_name = edf_name.replace('.edf', '_raw.fif')
                    fif_path = os.path.join(root, fif_name)
                    raw.save(fif_path, overwrite=True)
                    print(f"已保存为 {os.path.basename(fif_path)}")

                except Exception as e:
                    print(f"处理 {edf_name} 时出错: {e}")

# 主函数
if __name__ == "__main__":
    folder_path = r"data/chb-mit/chb01" # EDF 文件路径
    summary_path = r"data/chb-mit/chb01/chb01-summary.txt"  # 摘要文件路径

    annotate_edf_files(folder_path, summary_path)
    print("所有文件处理完成")

In [ ]:
import mne
import matplotlib.pyplot as plt

# 假设 raw 是你已经读取并添加注释后的 MNE Raw 对象
# 示例代码中仅作示意，你要根据实际情况替换 raw
#做这个可视化的目的就是为了验证是否成功注释 没有别的用意
raw = mne.io.read_raw_fif("data/chb-mit/chb01/chb01_03_raw.fif", preload=True)

# 可视化注释
raw.plot(duration=60, scalings='auto', block=True, title='EEG 发作注释可视化')
plt.show()